# Model: Random Forest

Owner: **Daniel**

## Imports

These are just some example ones, please feel free to remove or add any

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error

from sklearn.ensemble import RandomForestRegressor

MODEL_NAME = "random_forest"

## Load preprocessed data

Please load the same preprocessed train and test files.

Do NOT re-clean or re-derive features here. If something looks wrong, or you want to add another feature, please let the team know and we can add it to the csv that we will all use.

We want all the models to have the same data to have a fair comparison.

In [ ]:
DATA_DIR = Path("../../data/NSW")
RESULTS_DIR = Path("../../results")
RESULTS_DIR.mkdir(exist_ok=True)

# TODO: update these paths once the shared train/test files exist
train = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["DATETIME"]).set_index("DATETIME").sort_index()
test = pd.read_csv(DATA_DIR / "test.csv", parse_dates=["DATETIME"]).set_index("DATETIME").sort_index()

print(f"train: {train.index.min()} -> {train.index.max()}  ({len(train)} rows)")
print(f"test:  {test.index.min()} -> {test.index.max()}  ({len(test)} rows)")

## Features and target

You're free to use a subset of these, or engineer new features from them (e.g. lags, rolling averages, calendar features from `DATETIME`) — as the previous section also said, just don't add any additional raw features.

If you think something's missing, mention it to the team so everyone can decide whether to add it to the shared file.

In [ ]:
# TODO: everything here is a placeholder for now
TARGET = "TOTALDEMAND"
FEATURES = ["TEMPERATURE", "forecast_closest", "forecast_12hr_prior", "forecast_dayprior"]

## Training the model

In this section please train and implement the model you have been assigned. There are lots of available packages, i.e. sklearn, that should be easily implementable. The training model should only need to be a few lines of code.

In [ ]:
# TODO train the model

In [ ]:
# TODO: cross-validate on the training data only to choose hyperparameters
#
# Examples:
#
# from sklearn.model_selection import TimeSeriesSplit
#
# tscv = TimeSeriesSplit(n_splits=5)
# for fold, (train_idx, val_idx) in enumerate(tscv.split(train)):
#     X_train_fold, X_val_fold = train[FEATURES].iloc[train_idx], train[FEATURES].iloc[val_idx]
#     y_train_fold, y_val_fold = train[TARGET].iloc[train_idx], train[TARGET].iloc[val_idx]
#     # fit your model on (X_train_fold, y_train_fold), score it on (X_val_fold, y_val_fold)
#     # compare scores across folds/hyperparameters, then pick the best setting

## Validating the model

Produce a `predictions` array or Series covering the full test period, aligned to `test.index`.

Do NOT use this step to tune your model, repeatedly checking against the test set and adjusting based on it will cause you to overfit.

In [ ]:
# TODO: produce predictions
# Examples:
#
# sklearn-style (Random Forest / XGBoost / LightGBM):
# predictions = model.predict(test[FEATURES])
#
# SARIMAX:
# predictions = fitted_model.forecast(steps=len(test), exog=test[FEATURES])
#
# LSTM: build test windows the same way as training, predict, then inverse-transform
# back to MW before comparing against test[TARGET]
predictions = None

## Validation checklist

Every model notebook uses this same function, so the numbers are computed the same way for everyone:
- **RMSE**: root mean squared error, in MW
- **MAE**: mean absolute error, in MW
- **MAPE**: mean absolute percentage error, in %

In [ ]:
def evaluate(y_true, y_pred, model_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    metrics = {"model": model_name, "rmse": rmse, "mae": mae, "mape_pct": mape}
    print(metrics)
    return metrics


metrics = evaluate(test[TARGET], predictions, MODEL_NAME)

## Save results

This writes your predictions and metrics into the shared `results/` folder so they can be pulled together for the report. Don't change the file paths, `MODEL_NAME`, or column names below.

In [ ]:
pd.Series(predictions, index=test.index, name=MODEL_NAME).to_csv(RESULTS_DIR / f"{MODEL_NAME}_predictions.csv")

comparison_path = RESULTS_DIR / "model_comparison.csv"
this_run = pd.DataFrame([metrics])

if comparison_path.exists():
    existing = pd.read_csv(comparison_path)
    existing = existing[existing["model"] != MODEL_NAME]
    this_run = pd.concat([existing, this_run], ignore_index=True)

this_run.to_csv(comparison_path, index=False)
this_run